<a href="https://colab.research.google.com/github/didimae/1brc/blob/main/generar_parches_indices_Madrid_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive

To access files stored in your Google Drive, you need to mount it. This will prompt you to authenticate your Google account.

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
!rm -rf ./dataset/X/*
!rm -rf ./dataset/Y/*
print("Carpetas locales vaciadas y listas para la regeneración limpia.")

Carpetas locales vaciadas y listas para la regeneración limpia.


In [22]:
import os
import glob
import geopandas as gpd

# Update RUTA_ENTRADA to point to your Google Drive path
# Make sure your '.tif' files are in 'TFM/Indices Sentinel y Landsat' within your Google Drive.
RUTA_ENTRADA = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat"
RUTA_SALIDA_X = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/X/"
RUTA_SALIDA_Y = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/Y/"

# Re-create output directories in case they were created in the wrong place
# initially
os.makedirs(RUTA_SALIDA_X, exist_ok=True)
os.makedirs(RUTA_SALIDA_Y, exist_ok=True)

# Re-run the glob search for .tif files after updating RUTA_ENTRADA
archivos_tif = glob.glob(os.path.join(RUTA_ENTRADA, "Madrid_EPSG25830_5m_*.tif"))
print(f"Se han encontrado {len(archivos_tif)} archivos anuales para procesar.")

# Lee el shapefile desde la ruta de mi Drive
prueba_shp = gpd.read_file('/content/drive/MyDrive/TFM/edificios/ALTURAS_EDIFICIOS.shp')
#print(prueba_shp.columns)
COLUMNA_ALTURA = "ALTURA"

Se han encontrado 10 archivos anuales para procesar.


In [23]:
import os
import glob
import rasterio
from rasterio.features import rasterize
import geopandas as gpd
import numpy as np
from tqdm import tqdm

# =====================================================================
# CONFIGURACIÓN DE RUTAS Y PARÁMETROS
# =====================================================================
RUTA_TIFFS = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat"          # Carpeta con tus 10 .tif de Earth Engine
RUTA_SHAPEFILE = "/content/drive/MyDrive/TFM/edificios/ALTURAS_EDIFICIOS.shp" # Tu Shapefile de edificios
COLUMNA_ALTURA = "ALTURA"              # ¡Cambia esto por el nombre de tu columna de alturas!

RUTA_SALIDA_X = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/X/"         # Destino variables de entrada (4 canales)
RUTA_SALIDA_Y = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/Y/"         # Destino variable objetivo LST (1 canal)

TAMANO_PARCHE = 256  # 256 píxeles = 1.28 km
SOLAPE = 0.25        # 25% de solape
STRIDE = int(TAMANO_PARCHE * (1 - SOLAPE)) # Desplazamiento de 192 píxeles

os.makedirs(RUTA_SALIDA_X, exist_ok=True)
os.makedirs(RUTA_SALIDA_Y, exist_ok=True)

# =====================================================================
# 1. CARGAR Y COMPROBAR EL SHAPEFILE (Una sola vez)
# =====================================================================
print("Cargando Shapefile de morfología urbana de Madrid...")
edificios = gpd.read_file(RUTA_SHAPEFILE)

# Asegurar por seguridad que el Shapefile esté en la misma proyección oficial EPSG:25830
if edificios.crs.to_string() != "EPSG:25830":
    print("Reproyectando Shapefile automáticamente a EPSG:25830...")
    edificios = edificios.to_crs("EPSG:25830")

# =====================================================================
# 2. BUCLE PARA PROCESAR LOS 10 AÑOS Y FUSIONAR DATOS
# =====================================================================
archivos_tif = sorted(glob.glob(os.path.join(RUTA_TIFFS, "Madrid_EPSG25830_5m_*.tif")))
print(f"Se han encontrado {len(archivos_tif)} archivos anuales para procesar.")

contador_parches = 0

for ruta_img in archivos_tif:
    nombre_archivo = os.path.basename(ruta_img)
    ano = nombre_archivo.split("_")[-1].replace(".tif", "")
    print(f"\nFusionando e indexando año {ano}...")

    with rasterio.open(ruta_img) as src:
        transform = src.transform
        alto, ancho = src.shape

        # Leer los índices calculados en Earth Engine
        ndvi = src.read(1) # Canal 1 original (.tif)
        ndbi = src.read(2) # Canal 2 original (.tif)
        lst  = src.read(3) # Canal 3 original (.tif) -> Este será la Y independiente

        # --- RASTERIZACIÓN DE LA MORFOLOGÍA URBANA AL VUELO ---
        # Creamos el Canal 3: Huella binaria (1 edificio, 0 suelo libre)
        huella_raster = rasterize(
            [(geom, 1) for geom in edificios.geometry],
            out_shape=(alto, ancho),
            transform=transform,
            fill=0,
            dtype='float32'
        )

        # Creamos el Canal 4: Alturas reales extraídas del Shapefile
        alturas_raster = rasterize(
            [(geom, attr) for geom, attr in zip(edificios.geometry, edificios[COLUMNA_ALTURA])],
            out_shape=(alto, ancho),
            transform=transform,
            fill=0,
            dtype='float32'
        )

         # --- LÍNEA DE SEGURIDAD CRÍTICA PARA EL TFM ---
        # Todo lo que sea un valor de fondo negativo (NoData) o NaN se convierte en 0.0 metros reales (calles/plazas)
        alturas_raster = np.where((alturas_raster < 0) | (np.isnan(alturas_raster)), 0.0, alturas_raster)
        # Limitamos el techo por seguridad estadística (ej. si hay un error de tecleo en el SHP de 999 metros)
        alturas_raster = np.clip(alturas_raster, 0.0, 100.0)

        # Unimos las 4 entradas para la variable X de la U-Net
        # Forma resultante: (4, alto, ancho)
        matriz_X_completa = np.stack([ndvi, ndbi, huella_raster, alturas_raster], axis=0)

        # =====================================================================
        # 3. EXTRAER PARCHES CON VENTANA DESLIZANTE (STRIDE)
        # =====================================================================
        for y in range(0, alto - TAMANO_PARCHE + 1, STRIDE):
            for x in range(0, ancho - TAMANO_PARCHE + 1, STRIDE):

                # Cortar submatriz de 256x256
                parche_x = matriz_X_completa[:, y:y+TAMANO_PARCHE, x:x+TAMANO_PARCHE]
                parche_y = lst[y:y+TAMANO_PARCHE, x:x+TAMANO_PARCHE]

                # Control de calidad: descartar zonas fuera de la máscara de Madrid (NoData)
                if np.isnan(parche_x).any() or np.isnan(parche_y).any() or np.min(parche_y) <= 15:
                    continue

                # Adaptar dimensiones al formato estándar de TensorFlow:
                # X: (4, 256, 256) -> (256, 256, 4)  |  Y: (256, 256) -> (256, 256, 1)
                parche_x = np.moveaxis(parche_x, 0, -1)
                parche_y = np.expand_dims(parche_y, axis=-1)

                # Guardar los archivos binarios independientes
                id_parche = f"madrid_{ano}_p{contador_parches:05d}"
                np.save(os.path.join(RUTA_SALIDA_X, f"{id_parche}_X.npy"), parche_x.astype(np.float32))
                np.save(os.path.join(RUTA_SALIDA_Y, f"{id_parche}_Y.npy"), parche_y.astype(np.float32))

                contador_parches += 1

print(f"\n¡Fusión completada con éxito!")
print(f"Total de parches de 4 canales generados para tu U-Net: {contador_parches}")


Cargando Shapefile de morfología urbana de Madrid...
Se han encontrado 10 archivos anuales para procesar.

Fusionando e indexando año 2016...

Fusionando e indexando año 2017...

Fusionando e indexando año 2018...

Fusionando e indexando año 2019...

Fusionando e indexando año 2020...

Fusionando e indexando año 2021...

Fusionando e indexando año 2022...

Fusionando e indexando año 2023...

Fusionando e indexando año 2024...

Fusionando e indexando año 2025...

¡Fusión completada con éxito!
Total de parches de 4 canales generados para tu U-Net: 8405


Esta celda es el corazón del control de calidad del dataset y sirve para garantizar dos cosas fundamentales: la limpieza de datos corruptos y la sincronización perfecta entre las variables de entrada (X) y la temperatura objetivo (Y).
Su funcionamiento se divide en tres tareas críticas:
## 1. El Recorte Geográfico (# CORTAR AMBOS)
Las líneas parche_x = ... y parche_y = ... actúan como una "tijera digital". Utilizando las coordenadas de la ventana deslizante (x e y), extraen simultáneamente un cuadrado de 256 × 256 píxeles de la matriz grande de Madrid. Cortan el mismo trozo de ciudad exacta para los índices y edificios (X) que para la temperatura (Y).
## 2. El Filtro Anti-Contaminación (# FILTRAR AMBOS A LA VEZ)
La condición if not (...) es un escudo de seguridad que analiza el contenido de los dos parches antes de guardarlos. Solo los deja pasar si cumplen estas tres leyes físicas:

* np.isnan(parche_x).any(): Comprueba si hay píxeles vacíos (NaN) en los índices o alturas.
* np.isnan(parche_y).any(): Comprueba si hay píxeles vacíos en la temperatura de Landsat.
* np.min(parche_y) <= 0: Comprueba si la temperatura del parche cae por debajo de 0 °C (lo cual indicaría que es un píxel erróneo de NoData del satélite, ya que en Madrid en verano la LST al sol nunca baja de 15 °C o 20 °C).

Si cualquiera de estas tres condiciones es verdadera, el parche completo se descarta (gracias al filtro if not) y el script salta al siguiente trozo de ciudad. Así evitas que tu U-Net intente aprender de píxeles sin información o con errores de satélite.
## 3. El Guardado Simétrico y Formateo para TensorFlow
Si el parche supera con éxito el filtro de calidad, se ejecuta el bloque interno:

* Alineación de nombres: Genera un código único (id_parche) idéntico tanto para la entrada como para la salida (ej. madrid_2016_p00005_X.npy y madrid_2016_p00005_Y.npy). Esto garantiza que jamás se desalinee tu dataset y que el Data Loader los empareje a la perfección.
* np.moveaxis(parche_x, 0, -1): Reordena las dimensiones del parche X para pasarlo del formato SIG (Canales, Alto, Ancho) al formato nativo que exige TensorFlow: (Alto, Ancho, Canales) -> (256, 256, 4).
* np.expand_dims(parche_y, axis=-1): Añade una dimensión de profundidad al parche térmico Y para que pase de ser una matriz plana de (256, 256) a un cubo de un solo canal: (256, 256, 1).


In [24]:
# CORTAR AMBOS
parche_x = matriz_X_completa[:, y:y+TAMANO_PARCHE, x:x+TAMANO_PARCHE]
parche_y = lst[y:y+TAMANO_PARCHE, x:x+TAMANO_PARCHE]

# FILTRAR AMBOS A LA VEZ
# Si el parche tiene NaNs, o si la temperatura mínima es menor a 15ºC (ruido de nubes/sombras), lo descartamos
if not (np.isnan(parche_x).any() or np.isnan(parche_y).any() or np.min(parche_y) <= 15):
    # SOLO SI AMBOS SON VÁLIDOS, SE GUARDAN JUNTOS
    id_parche = f"madrid_{ano}_p{contador_parches:05d}"
    np.save(os.path.join(RUTA_SALIDA_X, f"{id_parche}_X.npy"), np.moveaxis(parche_x, 0, -1).astype(np.float32))
    np.save(os.path.join(RUTA_SALIDA_Y, f"{id_parche}_Y.npy"), np.expand_dims(parche_y, axis=-1).astype(np.float32))
    contador_parches += 1


In [25]:
import os
import glob

# Configuración de las rutas de tus carpetas locales en Colab
DIR_X = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/X/"
DIR_Y = "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/Y/"

print("=== INICIANDO LOGS DE AUDITORÍA DEL DATASET ===")

# 1. Escaneo bruto de los archivos en disco
rutas_brutas_x = glob.glob(os.path.join(DIR_X, "*_X.npy"))
rutas_brutas_y = glob.glob(os.path.join(DIR_Y, "*_Y.npy"))

print(f"[LOG] Archivos .npy encontrados físicamente en carpeta X: {len(rutas_brutas_x)}")
print(f"[LOG] Archivos .npy encontrados físicamente en carpeta Y: {len(rutas_brutas_y)}")

# 2. Extraer el identificador único (ID) de cada parche para poder cruzarlos
# Ejemplo: transforma './dataset/X/madrid_2016_p00005_X.npy' en 'madrid_2016_p00005'
set_id_x = {os.path.basename(f).replace("_X.npy", "") for f in rutas_brutas_x}
set_id_y = {os.path.basename(f).replace("_Y.npy", "") for f in rutas_brutas_y}

# 3. Calcular intersecciones y diferencias matemáticas
parches_correctos = set_id_x.intersection(set_id_y)
huerfanos_en_x = set_id_x - set_id_y
huerfanos_en_y = set_id_y - set_id_x

print(f"\n[LOG] Parches perfectos (tienen X e Y con el mismo nombre): {len(parches_correctos)}")
print(f"[LOG] ALERTA: Parches en X sin correspondencia térmica en Y: {len(huerfanos_en_x)}")
print(f"[LOG] ALERTA: Parches en Y sin correspondencia de variables en X: {len(huerfanos_en_y)}")

# 4. EJECUTAR PURGA Y TRAZAR EL BORRADO
if len(huerfanos_en_x) > 0:
    print("\n[PROCESO] Eliminando huérfanos de la carpeta X...")
    for h in huerfanos_en_x:
        os.remove(os.path.join(DIR_X, f"{h}_X.npy"))
    print("[LOG] Limpieza de X completada.")

if len(huerfanos_en_y) > 0:
    print("\n[PROCESO] Eliminando huérfanos de la carpeta Y...")
    for h in huerfanos_en_y:
        os.remove(os.path.join(DIR_Y, f"{h}_Y.npy"))
    print("[LOG] Limpieza de Y completada.")

print("\n=== COMPROBACIÓN FINAL DE ALINEACIÓN ===")
# Volvemos a leer tras la purga para verificar la simetría absoluta
archivos_finales_x = sorted(glob.glob(os.path.join(DIR_X, "*_X.npy")))
archivos_finales_y = sorted(glob.glob(os.path.join(DIR_Y, "*_Y.npy")))

print(f"[LOG] Total X final: {len(archivos_finales_x)}")
print(f"[LOG] Total Y final: {len(archivos_finales_y)}")

# Muestra un log visual de las 5 primeras parejas para verificar que coinciden año y parche
if len(archivos_finales_x) > 0 and len(archivos_finales_x) == len(archivos_finales_y):
    print("\n[LOG] Ejemplo de emparejamiento estricto indexado (Primeras 5 muestras):")
    for i in range(min(5, len(archivos_finales_x))):
        nombre_x = os.path.basename(archivos_finales_x[i])
        nombre_y = os.path.basename(archivos_finales_y[i])
        print(f"   Pareja [{i}]:  Variable_Input: {nombre_x}  <--->  Target_Thermal: {nombre_y}")
else:
    print("\n[ERROR CRÍTICO] Las carpetas siguen descompensadas. Revisa los permisos de lectura.")


=== INICIANDO LOGS DE AUDITORÍA DEL DATASET ===
[LOG] Archivos .npy encontrados físicamente en carpeta X: 8405
[LOG] Archivos .npy encontrados físicamente en carpeta Y: 8405

[LOG] Parches perfectos (tienen X e Y con el mismo nombre): 8405
[LOG] ALERTA: Parches en X sin correspondencia térmica en Y: 0
[LOG] ALERTA: Parches en Y sin correspondencia de variables en X: 0

=== COMPROBACIÓN FINAL DE ALINEACIÓN ===
[LOG] Total X final: 8405
[LOG] Total Y final: 8405

[LOG] Ejemplo de emparejamiento estricto indexado (Primeras 5 muestras):
   Pareja [0]:  Variable_Input: madrid_2016_p00000_X.npy  <--->  Target_Thermal: madrid_2016_p00000_Y.npy
   Pareja [1]:  Variable_Input: madrid_2016_p00001_X.npy  <--->  Target_Thermal: madrid_2016_p00001_Y.npy
   Pareja [2]:  Variable_Input: madrid_2016_p00002_X.npy  <--->  Target_Thermal: madrid_2016_p00002_Y.npy
   Pareja [3]:  Variable_Input: madrid_2016_p00003_X.npy  <--->  Target_Thermal: madrid_2016_p00003_Y.npy
   Pareja [4]:  Variable_Input: madri

Now, you can run the original script to process your `.tif` files from Google Drive.

## Celda de Auditoría Matemática y Control de Rangos

Este código escaneará los parches binarios .npy recién generados en el almacenamiento de Drive y verificará que ningún valor se salga de las leyes de la física urbana.

In [26]:
import os
import glob
import numpy as np

# Crear las carpetas locales de Colab
!mkdir -p ./dataset/X/
!mkdir -p ./dataset/Y/
DIR_X = "./dataset/X/"
DIR_Y = "./dataset/Y/"

# Montar Google Drive si aún no está montado (si es necesario)
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Copiar los archivos desde tu Drive al disco local rápido (Ajusta las rutas si tu carpeta en Drive se llama distinto)
print("Copiando variables de entrada X al almacenamiento local...")
!cp -r "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/X"/* ./dataset/X/

print("Copiando targets térmicos Y al almacenamiento local...")
!cp -r "/content/drive/MyDrive/TFM Indices Sentinel y Landsat/dataset/Y"/* ./dataset/Y/

print("¡Traspaso completado! Datos listos en el disco rápido.")

archivos_x = glob.glob(os.path.join(DIR_X, "*_X.npy"))
archivos_y = glob.glob(os.path.join(DIR_Y, "*_Y.npy"))

print("=== INICIANDO CONTROL DE CALIDAD Y RANGOS DEL DATASET ===")

# Inicializamos variables para capturar los extremos absolutos de todo tu dataset
min_ndvi, max_ndvi = float('inf'), float('-inf')
min_ndbi, max_ndbi = float('inf'), float('-inf')
min_huella, max_huella = float('inf'), float('-inf')
min_altura, max_altura = float('inf'), float('-inf')
min_lst, max_lst = float('inf'), float('-inf')

errores_detectados = 0

# Analizamos una muestra significativa de 500 parches aleatorios para no ralentizar Colab
muestra_control = np.random.choice(archivos_x, size=min(500, len(archivos_x)), replace=False)

for ruta_x in muestra_control:
    # Construimos la ruta de su pareja térmica Y
    ruta_y = ruta_x.replace("/X/", "/Y/").replace("_X.npy", "_Y.npy")

    parche_x = np.load(ruta_x) # Forma: (256, 256, 4)
    parche_y = np.load(ruta_y) # Forma: (256, 256, 1)

    # Extraemos los extremos de cada canal en este parche concreto
    min_ndvi = min(min_ndvi, np.min(parche_x[..., 0]))
    max_ndvi = max(max_ndvi, np.max(parche_x[..., 0]))

    min_ndbi = min(min_ndbi, np.min(parche_x[..., 1]))
    max_ndbi = max(max_ndbi, np.max(parche_x[..., 1]))

    min_huella = min(min_huella, np.min(parche_x[..., 2]))
    max_huella = max(max_huella, np.max(parche_x[..., 2]))

    min_altura = min(min_altura, np.min(parche_x[..., 3]))
    max_altura = max(max_altura, np.max(parche_x[..., 3]))

    min_lst = min(min_lst, np.min(parche_y))
    max_lst = max(max_lst, np.max(parche_y))

print(f"\n[CONTROL] Rango Canal 0 (NDVI):   [{min_ndvi:.4f} a {max_ndvi:.4f}]   -> Esperado: [-1 a 1]")
print(f"[CONTROL] Rango Canal 1 (NDBI):   [{min_ndbi:.4f} a {max_ndbi:.4f}]   -> Esperado: [-1 a 1]")
print(f"[CONTROL] Rango Canal 2 (Huella): [{min_huella:.1f} a {max_huella:.1f}]   -> Esperado: [0 a 1]")
print(f"[CONTROL] Rango Canal 3 (Altura): [{min_altura:.1f} a {max_altura:.1f}m]  -> Esperado: [0 a ~100m]")
print(f"[CONTROL] Rango Target  (LST):    [{min_lst:.2f}ºC a {max_lst:.2f}ºC] -> Esperado: [15ºC a 50ºC]")

# --- CRITERIOS DE CONTROL DE SEGURIDAD (TRAZAS DE ERROR) ---
if min_altura < 0:
    print("\n[ALERTA CRÍTICA] ¡ERROR! Se han detectado alturas negativas en los parches. El fondo -9999 sigue vivo.")
    errores_detectados += 1

if min_lst < 10:
    print("\n[ALERTA CRÍTICA] ¡ERROR! Temperaturas anormalmente frías en verano en Madrid. Revisa NoData de GEE.")
    errores_detectados += 1

if errores_detectados == 0:
    print("\n" + "="*60)
    print("¡VEREDICTO: DATASET VALIDADO! Cero valores NoData detectados.")
    print("El rango métrico de alturas es correcto. Listo para entrenar de forma segura.")
    print("="*60)
else:
    print(f"\n[FALLO] El dataset tiene {errores_detectados} anomalías estructurales. No inicies el entrenamiento.")

Copiando variables de entrada X al almacenamiento local...
Copiando targets térmicos Y al almacenamiento local...
¡Traspaso completado! Datos listos en el disco rápido.
=== INICIANDO CONTROL DE CALIDAD Y RANGOS DEL DATASET ===

[CONTROL] Rango Canal 0 (NDVI):   [-0.5834 a 0.9110]   -> Esperado: [-1 a 1]
[CONTROL] Rango Canal 1 (NDBI):   [-0.6873 a 0.7180]   -> Esperado: [-1 a 1]
[CONTROL] Rango Canal 2 (Huella): [0.0 a 1.0]   -> Esperado: [0 a 1]
[CONTROL] Rango Canal 3 (Altura): [0.0 a 100.0m]  -> Esperado: [0 a ~100m]
[CONTROL] Rango Target  (LST):    [15.09ºC a 61.31ºC] -> Esperado: [15ºC a 50ºC]

¡VEREDICTO: DATASET VALIDADO! Cero valores NoData detectados.
El rango métrico de alturas es correcto. Listo para entrenar de forma segura.
